In [50]:
from src import clustering, graph_utils
import numpy as np
import spatialdata as sd
import os 
from napari_spatialdata import Interactive
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import squidpy as sq


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
dirs = os.listdir("C://Users//laure//Desktop//github//sys_gen//zarr")

In [4]:
dirs

['human_bone_marrow_acute_lymphoid_leukemia',
 'human_brain_preview',
 'human_non_diseased_lung',
 'mouse_bone_formic_acid_decalcification',
 'mouse_brain_replicate_1',
 'mouse_colon']

In [4]:
sdata = sd.read_zarr("C://Users//laure//Desktop//github//sys_gen//zarr//" + dirs[4])
adata = sdata.tables["table"]

In [5]:
graph_utils.get_distances(adata, k=1, log=True)

In [6]:
results = clustering.fit_gmm(adata, distance_key="1_nn_distance", k_range=range(1,10))

Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\laure\anaconda3\envs\new_spatial\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\laure\anaconda3\envs\new_spatial\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\laure\anaconda3\envs\new_spatial\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\laure\anaconda3\envs\new_spatial\Lib\subprocess.py", line 1601, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0x81 in position 120: invalid start byte
c:\Users\laure\anaconda3\envs\new_spatial\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible a

Fitted GMM with k=1
Fitted GMM with k=2
Fitted GMM with k=3
Fitted GMM with k=4
Fitted GMM with k=5
Fitted GMM with k=6
Fitted GMM with k=7
Fitted GMM with k=8
Fitted GMM with k=9


In [7]:
best_model = clustering.choose_component(results, delta_bic_threshold=10.0)

In [13]:
adata.obs["gmm_labels"] = best_model["labels"]
adata.obs["gmm_labels"] = adata.obs["gmm_labels"].astype("category") 

In [14]:
graph_utils.get_neighbors(adata, type="delaunay", gmm_labels="gmm_labels")

In [15]:
graph_utils.prune_graph(adata, distance_key="1_nn_distance", type="percentile")

In [16]:
labels = graph_utils.set_seeds(adata, "1_nn_distance","gmm_labels", spatial_connectivity_key="spatial_connectivities_pruned")

In [17]:
adata.obs["seed_labels"] = labels

In [29]:
watershed_labels = clustering.watershed(adata, distances_key="1_nn_distance", spatial_connectivity_key="spatial_connectivities", labels_key="seed_labels")

In [30]:
sdata.tables["table"].obs["watershed_labels"] = watershed_labels
sdata.tables["table"].obs["watershed_labels"] = sdata.tables["table"].obs["watershed_labels"].astype("category")

In [43]:
def assign_random_colors(n, cmap_name="hsv", seed=42):
    """
    Assign n colors sampled from a continuous colormap in random order.
    Avoids grid-like bands for large n.
    """
    rng = np.random.default_rng(seed)
    
    # Random positions in [0,1)
    indices = rng.random(n)
    
    cmap = cm.get_cmap(cmap_name)
    colors = [mcolors.to_hex(cmap(i)) for i in indices]
    
    return colors

In [47]:
n_labels = sdata.tables["table"].obs["watershed_labels"].nunique()
sdata.tables["table"].uns["watershed_labels_colors"] = assign_random_colors(n_labels, cmap_name="tab20")


C:\Users\laure\AppData\Local\Temp\ipykernel_5492\1396928867.py:11: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap(cmap_name)


In [48]:
sdata.tables["table"].obs["watershed_labels"].nunique()

10683

In [ ]:
Interactive(sdata)

2026-01-26 17:03:31.602 | WARNING  | napari_spatialdata._viewer:__init__:57 - Due to Shift-L being used as shortcut in napari, it is being deprecated and might not link a new layer to an existing SpatialData object in the viewer. Please use ⌘-L on MacOS or else Ctrl-L.


2026-01-26 17:03:34.447 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2026-01-26 17:03:34.694 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
